### 1. Exploring the Jewish Concert Archive JSON

This notebook is a guided tour of `jewish_concert_archive.json` — a hand-curated catalog of concerts, revues, and theatrical performances documented in the digitized papers of violinist/orchestra leader Majer "Ivan" Pietruschka (see the archive's own `archive_overview` field, loaded below, for the full story).

They come from the Melbourne Jewish Museum's collection materials from the Dunera voyage and the internment of Jewish refugees in Australia during World War II. The JSON is a nested structure, with each concert record containing multiple fields, some of which are optional or inconsistently named.

The goals for this workshop session are to:

1. Understand how the JSON is actually structured (which is *not* a single flat table).
2. Harvest four things from it: **locations**, **dates**, **event types**, and **persons involved (with their roles)**.
3. Build a tidy `pandas` table and make some exploratory bar charts and scatter plots with **Plotly Express**.
4. Build **bi-nodal Person–Place network**, which we'll construct using `networkx` and `pyvis`.

Throughout, the emphasis is on showing the *technique* for harvesting messy, real-world nested JSON — not just the final numbers. Several of the choices below (how we bucket event types, how we shorten location names) are simple heuristics meant to be inspected and adjusted, not treated as ground truth.

### 2.  Import Libraries

In [250]:
import json

from collections import Counter

import pandas as pd
import plotly.express as px


### 3. Loading the archive

At the top level, the JSON is a dictionary with just two keys:

- `archive_overview` — a single prose string describing the collection as a whole.
- `concerts` — a list of dictionaries, one per cataloged event (despite the key's name, this includes concerts, revues, plays, and even a few standalone songs — see below).  

There is no schema file enforcing what's inside each concert dictionary — it was built by a human cataloger working from archival documents, so the fields present vary from record to record depending on what that particular document actually contained. Handling that variability *is* the main exercise in this notebook.

In [251]:
with open("jewish_concert_archive.json") as f:
    archive = json.load(f)

concerts = archive["concerts"]

print(archive["archive_overview"])
print(f"\n{len(concerts)} cataloged events.")


This JSON catalogs concert, revue, and theatrical performance programs found in a digitized collection of personal papers belonging to (or collected by) Majer 'Ivan' Pietruschka, a Polish-born violinist and orchestra leader. Pietruschka played in the Warsaw and Lodz Symphony Orchestras and conducted a silent-film orchestra in Berlin (1923-1932) before Nazi restrictions on Jewish musicians drove him to England in 1939. In 1940 the British Government transported him and other refugees to Australia aboard the HMT/HMAT Dunera (attacked three times by German U-boats en route), where he was interned as an 'enemy alien,' first apparently at Hay Camp (NSW) and later at Tatura (Victoria). After release, he joined the Australian Army's 8th Employment Company, entertaining troops with an orchestra he formed, and after the war played with the 3DB Orchestra and, from 1952, the Melbourne Symphony Orchestra. He married pianist-composer Phyllis Batchelor in 1946 and settled in Heidelberg, Victoria. Ma

### 4. Anatomy of a single record

Let's look at one full record to see what a concert dictionary actually contains, then list just its keys for something easier to scan.

In [252]:
example = concerts[0]
# just the first x characters of the JSON for readability
print(json.dumps(example, indent=2)[:10000], "...\n")
print("Keys in this record:", sorted(example.keys()))


{
  "title": "Snowhite Joins Up",
  "alternate_titles": [
    "Snowhite: A Merry Xmas-New Year Revue"
  ],
  "type": "musical revue (pantomime/topical satire)",
  "venue": "Camp theatre (exact hall not specified)",
  "location": "Hay Internment Camp, New South Wales, Australia",
  "date": "1941",
  "presented_by": "Internees of Hay Internment Camp",
  "overall_credits": {
    "written_produced_directed_by": "Doc K. Sternberg",
    "music": "Ray Martin",
    "musical_arrangements_and_piano": [
      "Jonny Flynn",
      "Rudolf Laqueur",
      "Herbert Voss"
    ],
    "drums": "Kurt Mayer",
    "scenery": [
      "Emil Wittenberg",
      "Klaus Friedeberger",
      "Fritz Schoenbach",
      "Heinz Tichauer"
    ],
    "costumes": [
      "George Blank",
      "Willy Herr",
      "Kurt Rosenberg"
    ],
    "choreography": "Klaus Begach",
    "masks": "Bernhard Joseph",
    "stage_manager": "David Rummelsburg",
    "assistant_stage_manager": "H. P. Kessler",
    "technical_staff": [
   

### 5. JSON uses Key-Value Pairs

You can think of a JSON object as a dictionary, with keys and values. The keys are strings, and the values can be strings, numbers, booleans, lists, or even other dictionaries. In this case, each concert record is a dictionary with various keys representing different pieces of information about the event.

- NB:  The keys must be unique! The values can repeat.

Here we find all the keys in the first concert record. 



- Note also the inconsistent naming: `Complete credits` (capitalized, space) sits alongside `snake_case` keys like `overall_credits` — a small but real reminder to `.get()` fields by their *exact* key rather than guessing a convention.

#### 5a.  List of Keys

In [253]:
key_list = [key for key in concerts[0].keys()]
key_list

['title',
 'alternate_titles',
 'type',
 'venue',
 'location',
 'date',
 'presented_by',
 'overall_credits',
 'acts',
 'additional_notes',
 'Complete credits']

#### 5b. Counting the Keys in ALL the Records

What are the top level keys?  And are they evenly distributed across the set?

Here we use a Python `Counter` to count the number of times each key appears across all concert records. This will help us understand which keys are common and which are rare, and will inform how we handle missing data in our analysis.

- A handful of fields (`title`, `type`, `venue`, `location`, `date`, `additional_notes`, `Complete credits`) appear in every record — these form a reliable core. 

- Everything else (`overall_credits`, `acts`, `performers`, `cast`, `lyrics`, ...) is present only when the source document actually had that kind of information, so any code that harvests from them needs to check for presence rather than assume it.


In [254]:
key_counts = Counter()
for c in concerts:
    key_counts.update(c.keys())

print(f"Field frequency across all {len(concerts)} records:")
for key, count in key_counts.most_common():
    print(f"  {key:<30} {count}/{len(concerts)}")


Field frequency across all 26 records:
  title                          26/26
  type                           26/26
  venue                          26/26
  location                       26/26
  date                           26/26
  additional_notes               26/26
  Complete credits               26/26
  acts                           24/26
  overall_credits                15/26
  presented_by                   9/26
  performers                     6/26
  alternate_titles               4/26
  composer_lyricist              2/26
  lyrics                         2/26
  performers_list                2/26
  cast                           2/26


### 6. Working with Individual Keys

Here is now we get the value of a single key in the first concert record. For example, to get the date of the first concert, we can access the `date` key like this:

```python

selected_concert = concerts[0]  # get the first concert record
selected_concert['date']  # returns the date of the first concert
```

**Note**:  The `date` field is a string.  We could instead use the `datetime` library to convert it to a `datetime` object, which would allow us to do date arithmetic and comparisons, for example:  sorting by month and year, or filtering for concerts that occurred before or after a certain date.  For now, however, we will leave it as a string.

In [255]:
selected_concert = concerts[0]
selected_concert['date']

'1941'

### 7. Nested Keys

One of the challenges of working with JSON is that the values of **keys** can themselves be **dictionaries** or **lists**, which can themselves contain more dictionaries or lists. This is called "nesting," and it can go several levels deep. For example the `overall_credits` key includes a long series of sub_keys, each of which might be a single string, a list of strings, or even another dictionary. 

Here's an example of what the `overall_credits` key looks like for one record:
 
```python
"overall_credits": {
    "written_produced_directed_by": "Doc K. Sternberg",
    "music": "Ray Martin",
    "musical_arrangements_and_piano": [
      "Jonny Flynn",
      "Rudolf Laqueur",
      "Herbert Voss"
    ],
    "drums": "Kurt Mayer",
    "scenery": [
      "Emil Wittenberg",
      "Klaus Friedeberger",
      "Fritz Schoenbach",
      "Heinz Tichauer"
    ],
    "costumes": [
      "George Blank",
      "Willy Herr",
      "Kurt Rosenberg"
    ],
    "choreography": "Klaus Begach",
    "masks": "Bernhard Joseph",
    "stage_manager": "David Rummelsburg",
    "assistant_stage_manager": "H. P. Kessler",
    "technical_staff": [
      "Karl Bazant",
      "Alfred Deutsch",
      "Kurt Gruenbaum",
      "Bernhard Joseph",
      "Siegfried Lehmann"
    ]
  }
  ```

#### Some Nested Key Examples

Here we explore the keys that are nested within the `overall_credits` dictionary for this record. Note that other concert records may have different keys in their `overall_credits` dictionaries, so this is not a universal list of keys!

As a reminder, here are the keys we found above for this record:

```python
['title',
 'alternate_titles',
 'type',
 'venue',
 'location',
 'date',
 'presented_by',
 'overall_credits',
 'acts',
 'additional_notes',
 'Complete credits']
 ```


In [256]:
# 'music' is a string:'
selected_concert = concerts[0]
selected_concert['overall_credits']['music']

'Ray Martin'

In [257]:
# 'musical_arrangements_and_piano' is a list of strings:
selected_concert['overall_credits']['musical_arrangements_and_piano']

['Jonny Flynn', 'Rudolf Laqueur', 'Herbert Voss']

In [258]:
# meanwhile `Complete Credits` is a separate key listing all personnel for that concert
# note that t
selected_concert['Complete credits']

['Doc K. Sternberg',
 'Ray Martin',
 'Jonny Flynn',
 'Rudolf Laqueur',
 'Herbert Voss',
 'Kurt Mayer',
 'Emil Wittenberg',
 'Klaus Friedeberger',
 'Fritz Schoenbach',
 'Heinz Tichauer',
 'George Blank',
 'Willy Herr',
 'Kurt Rosenberg',
 'Klaus Begach',
 'Bernhard Joseph',
 'David Rummelsburg',
 'H. P. Kessler',
 'Karl Bazant',
 'Alfred Deutsch',
 'Kurt Gruenbaum',
 'Siegfried Lehmann',
 'O. H. Mayer',
 'Hugo Schuster',
 'Eric Liffmann',
 'H. R. Risotto',
 'Baron de Reissn[er]',
 'Ernest Hutterer',
 'Josef Almas',
 'Alfred Jablonsky',
 'Mops Major',
 'Klaus Presser',
 'Fred Rosenthal',
 'Erich Schlesinger',
 'Armin Stern',
 'Peter Alsberg',
 'Kurt Flussmann',
 'Heinz Hermannsohn',
 'Werner Hirschfeld',
 'Heinz Lippmann',
 'Fritz Gottfurcht',
 'Rudolf Popper',
 'Egon Lehrburger',
 'Alan Grey',
 'Felix Popper',
 'Frederic Hollaender',
 'Leo Bieber',
 'Kurt Baier',
 'Edward Nelken',
 'Fritz Weidenbaum']

We will need to anticipate these differences as we work with the data, and write code that can handle the variability in structure. 



### 8. From JSON to Pandas Dataframe:  Clean and Tidy Data.

Pandas is a powerful library for data manipulation and analysis. It provides data structures like DataFrames that are perfect for working with tabular data. In this section, we will convert our JSON data into a Pandas DataFrame, which will allow us to easily manipulate and analyze the concert records.

We can easily load our JSON data, flattening out the nested structures into a tabular format that Pandas can work with. This will involve extracting the relevant fields from each concert record and organizing them into columns in the DataFrame.  Then we can start to clean and tidy the data, making it ready for analysis.

We could simply load the JSON into a dataframe with `pd.DataFrame(concerts)`, but that would give us a single column for the nested `overall_credits` dictionary, which is not very useful. Instead, we'll use `json_normalize()` to flatten the nested structure and create separate columns for each key in `overall_credits`.




In [259]:
concerts_df = pd.json_normalize(concerts, sep="_")

# the 'head' is just the first few rows of the dataframe, for a quick look at the data
concerts_df.head()

,title,alternate_titles,type,venue,location,date,presented_by,acts,additional_notes,Complete credits,...,overall_credits_largs_bay_band_mandoline,overall_credits_largs_bay_band_banjo,overall_credits_music_and_sound_effects,overall_credits_production_seduction_abduction_by,"cast_Sir Herbert Hardrow, Bt (the noted French co-respondent)",cast_The Hon. Hilary Hardrow (author of 'Silly with the Slops'),cast_Hilda Murdock (lecturer on Mrs. Beeton's methods),"cast_The Mill Girl, alias 'The Watchwoman'",cast_Baggs (fresh from driving the coach down 'Dead Man's Gulch'),"cast_Murgatroyd Murdock, alias the gum-shoe merchant"
0,Snowhite Joins Up,[Snowhite: A Merry Xmas-New Year Revue],musical revue (pantomime/topical satire),Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",1941,Internees of Hay Internment Camp,"[{'act_name': 'Part I', 'songs': [{'number': '...",This is the direct predecessor of the 1943 Mel...,"[Doc K. Sternberg, Ray Martin, Jonny Flynn, Ru...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Don't Release Me, Mr. Layton (song manuscript)",NaN,standalone original song (handwritten manuscript),None,"Hay Internment Camp, New South Wales, Australia",1941,NaN,NaN,A satirical protest song by internees at Hay C...,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Die Geschichte vom braven Soldaten Schwejk (Th...,NaN,theatrical production with orchestra,None,"Tatura Internment Camp, Victoria, Australia",1941,Internees of Tatura Internment Camp,"[{'act_name': 'Prolog', 'songs': [{'title': 'P...",A full-length dramatic adaptation of Hašek's s...,"[Rolf Stein, Josef Almas, Emil Wittenberg, H. ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Wir Reisen um die Welt / We Travel Round the W...,"[Eine Revue in 9 Bildern, A Revue Without a Gi...",musical revue,None,"Tatura Internment Camp, Victoria, Australia",1941,NaN,[{'act_name': 'Full cast (German and English p...,"This revue, structured as a comic 'world tour'...","[Robert Mass, Ernst Mass, H. W. Katz, P. E. Sc...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Grosses Unterhaltungskonzert (Grand Entertainm...,NaN,concert,Camp concert hall (venue name not specified),"Tatura Internment Camp, Victoria, Australia",1942,Das Camp-Orchester (The Camp Orchestra),[{'act_name': 'Teil 1 (Part 1) - At the Piano:...,One of several fully-documented Tatura camp co...,"[M. Pietruschka, J. Chlumecky, O. Silberstein,...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [260]:
# there are now MANY new subcolumns from our un-nested JSON!

list(concerts_df.columns)

['title',
 'alternate_titles',
 'type',
 'venue',
 'location',
 'date',
 'presented_by',
 'acts',
 'additional_notes',
 'Complete credits',
 'overall_credits_written_produced_directed_by',
 'overall_credits_music',
 'overall_credits_musical_arrangements_and_piano',
 'overall_credits_drums',
 'overall_credits_scenery',
 'overall_credits_costumes',
 'overall_credits_choreography',
 'overall_credits_masks',
 'overall_credits_stage_manager',
 'overall_credits_assistant_stage_manager',
 'overall_credits_technical_staff',
 'composer_lyricist',
 'lyrics',
 'overall_credits_based_on',
 'overall_credits_format',
 'overall_credits_staging_and_title_role',
 'overall_credits_stage_and_set_design',
 'overall_credits_music_using_czech_folk_songs',
 'overall_credits_orchestra_direction',
 'overall_credits_assistant_director',
 'overall_credits_orchestra_members',
 'overall_credits_collaborators',
 'overall_credits_idea',
 'overall_credits_produced_by_director',
 'overall_credits_dances_by',
 'overall

Let's focus on four target fields one at a time: **type**, **date**, **location**, and **persons + roles**.

For those we want:


```python
selected_cols = ['title',
'date',
'venue',
'location',
'Complete credits',
'overall_credits_written_produced_directed_by',
'overall_credits_music',
'overall_credits_musical_arrangements_and_piano']
 ```




In [261]:
selected_cols = ['title',
'type',
'date',
'venue',
'location',
'Complete credits',
'overall_credits_written_produced_directed_by',
'overall_credits_music',
'overall_credits_musical_arrangements_and_piano']

concert_df_brief = concerts_df[selected_cols].copy()
concert_df_brief

,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia","[Doc K. Sternberg, Ray Martin, Jonny Flynn, Ru...",Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]"
1,"Don't Release Me, Mr. Layton (song manuscript)",standalone original song (handwritten manuscript),1941,None,"Hay Internment Camp, New South Wales, Australia",[],NaN,NaN,NaN
2,Die Geschichte vom braven Soldaten Schwejk (Th...,theatrical production with orchestra,1941,None,"Tatura Internment Camp, Victoria, Australia","[Rolf Stein, Josef Almas, Emil Wittenberg, H. ...",NaN,NaN,NaN
3,Wir Reisen um die Welt / We Travel Round the W...,musical revue,1941,None,"Tatura Internment Camp, Victoria, Australia","[Robert Mass, Ernst Mass, H. W. Katz, P. E. Sc...",NaN,H. W. Katz,NaN
4,Grosses Unterhaltungskonzert (Grand Entertainm...,concert,1942,Camp concert hall (venue name not specified),"Tatura Internment Camp, Victoria, Australia","[M. Pietruschka, J. Chlumecky, O. Silberstein,...",NaN,NaN,NaN
5,Arien Abend (Aria Evening),vocal recital,1941,None,"Tatura Internment Camp, Victoria, Australia","[Günter Hirschberg, Gerhard Hamburger]",NaN,NaN,NaN
6,M. Pietruschka Chamber Concert (untitled progr...,chamber concert,1940,None,"Hay Internment Camp, New South Wales, Australia","[M. Pietruschka, S. Cohn, Emil Wittenberg, Ehr...",NaN,NaN,NaN
7,1st Concert (Recreation Department),concert,1940,None,Early Internment Camp,"[M. Pietruschka, Franz Stampfl, W.A.B., H. W. ...",NaN,NaN,NaN
8,Sergeant Snow White,musical revue (three-act pantomime/wartime sat...,1943,"Union Theatre, University of Melbourne (Univer...","Melbourne, Victoria, Australia","[Doc K. Sternberg, A. P. Schmitz, Max Lewinsky...",NaN,NaN,NaN
9,Journey's End,full-length play,Not specified,None,Not specified,"[M. Bittermann, Bernard Joseph, David Rosentha...",NaN,NaN,NaN


### 9. Cleaning Data

It's often said that 80% of a data-analysis project involved cleaning up the messy or inconsistent data in your set.  Here we will look at few case-studies in how this works:

- Event types (creating a kind of 'super group' that will serve as a heading for related events that seem part of the same category)
- Date types (looking at how we can turn the current 'string' years into real date-time objects)
- Location (regularizing the various terms to simplify our charts and graphs)
- Persons within the various new 'role' columns (we have been very selective here, so far focusing only on the 'Complete credits', and the main composers and arrangers)

#### 9a Event type

`type` is free text written by the cataloger for each document, not a controlled vocabulary — so there are as many distinct `type` strings as there are shades of "concert" in the collection. Let's look at them all before deciding how to group them.


```python

# turn the 'type' column into a list of unique values
list(concert_df_brief['type'].unique())

# show the list
['musical revue (pantomime/topical satire)',
 'standalone original song (handwritten manuscript)',
 'theatrical production with orchestra',
 'musical revue',
 'concert',
 'vocal recital',
 'chamber concert',
 'musical revue (three-act pantomime/wartime satire)',
 'full-length play',
 'concert (chamber recital)',
 'concert (piano & vocal recital)',
 'piano recital',
 "play (theatrical production, likely Noel Coward's 'Hay Fever')",
 'variety show',
 'communal religious/cultural song evening (songbook)',
 'concert (orchestral, with organ)',
 'shipboard variety revue',
 'shipboard variety concert with one-act play',
 'shipboard variety show',
 'standalone original song (unit marching song)',
 'comic amateur play (one-act parody melodrama)',
 'standalone original songs (handwritten manuscript)']
 ```

 **Grouping Event Types**

Here we can 'map' the raw `type` strings to a smaller set of categories. A dictionary makes every decision explicit and inspectable, and is easy to adjust if we spot a miscategorized value below.

This will create a new key in each record called `event_category`, which we can use for analysis and visualization.

In [262]:
# here we 
TYPE_CATEGORY_MAP = {
    "chamber concert": "Concert",
    "comic amateur play (one-act parody melodrama)": "Theatrical",
    "communal religious/cultural song evening (songbook)": "Other",
    "concert": "Concert",
    "concert (chamber recital)": "Concert",
    "concert (orchestral, with organ)": "Concert",
    "concert (piano & vocal recital)": "Concert",
    "full-length play": "Theatrical",
    "musical revue": "Revue/Variety",
    "musical revue (pantomime/topical satire)": "Revue/Variety",
    "musical revue (three-act pantomime/wartime satire)": "Revue/Variety",
    "piano recital": "Concert",
    "play (theatrical production, likely noel coward's 'hay fever')": "Theatrical",
    "shipboard variety concert with one-act play": "Revue/Variety",
    "shipboard variety revue": "Revue/Variety",
    "shipboard variety show": "Revue/Variety",
    "standalone original song (handwritten manuscript)": "Other",
    "standalone original song (unit marching song)": "Other",
    "standalone original songs (handwritten manuscript)": "Other",
    "theatrical production with orchestra": "Theatrical",
    "variety show": "Revue/Variety",
    "vocal recital": "Concert",
}

# now map the `type` column to a new `type_category` column using the dictionary
concert_df_brief['type_category'] = concert_df_brief['type'].map(TYPE_CATEGORY_MAP)
concert_df_brief.head()

,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano,type_category
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia","[Doc K. Sternberg, Ray Martin, Jonny Flynn, Ru...",Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety
1,"Don't Release Me, Mr. Layton (song manuscript)",standalone original song (handwritten manuscript),1941,None,"Hay Internment Camp, New South Wales, Australia",[],NaN,NaN,NaN,Other
2,Die Geschichte vom braven Soldaten Schwejk (Th...,theatrical production with orchestra,1941,None,"Tatura Internment Camp, Victoria, Australia","[Rolf Stein, Josef Almas, Emil Wittenberg, H. ...",NaN,NaN,NaN,Theatrical
3,Wir Reisen um die Welt / We Travel Round the W...,musical revue,1941,None,"Tatura Internment Camp, Victoria, Australia","[Robert Mass, Ernst Mass, H. W. Katz, P. E. Sc...",NaN,H. W. Katz,NaN,Revue/Variety
4,Grosses Unterhaltungskonzert (Grand Entertainm...,concert,1942,Camp concert hall (venue name not specified),"Tatura Internment Camp, Victoria, Australia","[M. Pietruschka, J. Chlumecky, O. Silberstein,...",NaN,NaN,NaN,Concert


### 9b.  Date

Dates in this collection are just bare years (some events couldn't be dated more precisely than that), or the literal string `"Not specified"` when even the year is unknown. We'll parse what we can into an integer year and leave the rest as missing, rather than guessing.

Note:  We will need to decide what to do about events that lack dates!

In [263]:
# check the unique values in the `date` column to see what kinds of date formats we have

concert_df_brief['date'].unique()


array(['1941', '1942', '1940', '1943', 'Not specified', '1944', '1939'],
      dtype=object)

In [264]:
# counts of each date, so we understand the distribution of events

# clearly there are significant number of events for which we lack date information at all!

concert_df_brief['date'].value_counts()

date
1941             11
Not specified     6
1940              3
1943              3
1942              1
1944              1
1939              1
Name: count, dtype: int64

**Date format?**

Currently the date field is formatted as a 'str'.  Here '1941' is simply a string of characters--not a number.  

```python
# check the type:

type(concert_df_brief['date'][0])
str
```



But if we wanted to deal with this more intelligently, perhaps as date-time format, which would allow us to group events by decade, month, etc. 

Note that we don't have that detail here, but it's good to keep in mind.

In [265]:
# create new date-time column based on the existing one

concert_df_brief['date_time'] = pd.to_datetime(concert_df_brief['date'], errors='coerce')
concert_df_brief.head(10)


,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano,type_category,date_time
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia","[Doc K. Sternberg, Ray Martin, Jonny Flynn, Ru...",Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01
1,"Don't Release Me, Mr. Layton (song manuscript)",standalone original song (handwritten manuscript),1941,None,"Hay Internment Camp, New South Wales, Australia",[],NaN,NaN,NaN,Other,1941-01-01
2,Die Geschichte vom braven Soldaten Schwejk (Th...,theatrical production with orchestra,1941,None,"Tatura Internment Camp, Victoria, Australia","[Rolf Stein, Josef Almas, Emil Wittenberg, H. ...",NaN,NaN,NaN,Theatrical,1941-01-01
3,Wir Reisen um die Welt / We Travel Round the W...,musical revue,1941,None,"Tatura Internment Camp, Victoria, Australia","[Robert Mass, Ernst Mass, H. W. Katz, P. E. Sc...",NaN,H. W. Katz,NaN,Revue/Variety,1941-01-01
4,Grosses Unterhaltungskonzert (Grand Entertainm...,concert,1942,Camp concert hall (venue name not specified),"Tatura Internment Camp, Victoria, Australia","[M. Pietruschka, J. Chlumecky, O. Silberstein,...",NaN,NaN,NaN,Concert,1942-01-01
5,Arien Abend (Aria Evening),vocal recital,1941,None,"Tatura Internment Camp, Victoria, Australia","[Günter Hirschberg, Gerhard Hamburger]",NaN,NaN,NaN,Concert,1941-01-01
6,M. Pietruschka Chamber Concert (untitled progr...,chamber concert,1940,None,"Hay Internment Camp, New South Wales, Australia","[M. Pietruschka, S. Cohn, Emil Wittenberg, Ehr...",NaN,NaN,NaN,Concert,1940-01-01
7,1st Concert (Recreation Department),concert,1940,None,Early Internment Camp,"[M. Pietruschka, Franz Stampfl, W.A.B., H. W. ...",NaN,NaN,NaN,Concert,1940-01-01
8,Sergeant Snow White,musical revue (three-act pantomime/wartime sat...,1943,"Union Theatre, University of Melbourne (Univer...","Melbourne, Victoria, Australia","[Doc K. Sternberg, A. P. Schmitz, Max Lewinsky...",NaN,NaN,NaN,Revue/Variety,1943-01-01
9,Journey's End,full-length play,Not specified,None,Not specified,"[M. Bittermann, Bernard Joseph, David Rosentha...",NaN,NaN,NaN,Theatrical,NaT


### 9c. Location

`location` is also free text, and it mixes several levels of geography into one string — e.g. `"Hay Internment Camp, New South Wales, Australia"`. 

- We might imagine **splitting** these into new columns for countries, regions, cities.

- Also, for our chart labels we don't want the full string, so we'll take everything before the first comma as a short place label.

- Watch what this reveals: `"Unspecified Internment camp"`, `"Unspecified internment camp"`, and `"An unspecified internment/military camp"` are three *different* strings for what's probably the same real-world "unknown camp" idea — a good example of why place names usually need a normalization/canonicalization pass before they become nodes in a network. 

We're flagging it here rather than silently fixing it, since this something for domain experts to decide! You could use a MAP (see the technique used above for working with the event categories) to do the job!

In [266]:
list(concert_df_brief['location'].unique())


['Hay Internment Camp, New South Wales, Australia',
 'Tatura Internment Camp, Victoria, Australia',
 'Early Internment Camp',
 'Melbourne, Victoria, Australia',
 'Not specified',
 'Unspecified Internment camp',
 'Unspecified internment camp',
 'An unspecified internment/military camp',
 'Sandwich, Kent, England',
 "Aboard His Majesty's Transport D3 (troopship)",
 'Aboard T.S.S. "Largs Bay" (Captain T. V. Roberts, R.D., R.N.R.)']

In [267]:
dict.fromkeys(list(concert_df_brief['location'].unique()))

{'Hay Internment Camp, New South Wales, Australia': None,
 'Tatura Internment Camp, Victoria, Australia': None,
 'Early Internment Camp': None,
 'Melbourne, Victoria, Australia': None,
 'Not specified': None,
 'Unspecified Internment camp': None,
 'Unspecified internment camp': None,
 'An unspecified internment/military camp': None,
 'Sandwich, Kent, England': None,
 "Aboard His Majesty's Transport D3 (troopship)": None,
 'Aboard T.S.S. "Largs Bay" (Captain T. V. Roberts, R.D., R.N.R.)': None}

In [268]:
# Your turn:  We could 'map' these names to something more useful!
location_dict = {'Hay Internment Camp, New South Wales, Australia': 'Hay',
 'Tatura Internment Camp, Victoria, Australia': 'Tatura',
 'Early Internment Camp': 'Other Camp',
 'Melbourne, Victoria, Australia': 'Melbourne',
 'Not specified': None,
 'Unspecified Internment camp': 'Other Camp',
 'Unspecified internment camp': 'Other Camp',
 'An unspecified internment/military camp': 'Other Camp',
 'Sandwich, Kent, England': "Sandwich",
 "Aboard His Majesty's Transport D3 (troopship)": "Dunera",
 'Aboard T.S.S. "Largs Bay" (Captain T. V. Roberts, R.D., R.N.R.)': "Larg's Bay"}

concert_df_brief['place_cleaned'] = concert_df_brief['location'].map(location_dict)
concert_df_brief

,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano,type_category,date_time,place_cleaned
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia","[Doc K. Sternberg, Ray Martin, Jonny Flynn, Ru...",Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
1,"Don't Release Me, Mr. Layton (song manuscript)",standalone original song (handwritten manuscript),1941,None,"Hay Internment Camp, New South Wales, Australia",[],NaN,NaN,NaN,Other,1941-01-01,Hay
2,Die Geschichte vom braven Soldaten Schwejk (Th...,theatrical production with orchestra,1941,None,"Tatura Internment Camp, Victoria, Australia","[Rolf Stein, Josef Almas, Emil Wittenberg, H. ...",NaN,NaN,NaN,Theatrical,1941-01-01,Tatura
3,Wir Reisen um die Welt / We Travel Round the W...,musical revue,1941,None,"Tatura Internment Camp, Victoria, Australia","[Robert Mass, Ernst Mass, H. W. Katz, P. E. Sc...",NaN,H. W. Katz,NaN,Revue/Variety,1941-01-01,Tatura
4,Grosses Unterhaltungskonzert (Grand Entertainm...,concert,1942,Camp concert hall (venue name not specified),"Tatura Internment Camp, Victoria, Australia","[M. Pietruschka, J. Chlumecky, O. Silberstein,...",NaN,NaN,NaN,Concert,1942-01-01,Tatura
5,Arien Abend (Aria Evening),vocal recital,1941,None,"Tatura Internment Camp, Victoria, Australia","[Günter Hirschberg, Gerhard Hamburger]",NaN,NaN,NaN,Concert,1941-01-01,Tatura
6,M. Pietruschka Chamber Concert (untitled progr...,chamber concert,1940,None,"Hay Internment Camp, New South Wales, Australia","[M. Pietruschka, S. Cohn, Emil Wittenberg, Ehr...",NaN,NaN,NaN,Concert,1940-01-01,Hay
7,1st Concert (Recreation Department),concert,1940,None,Early Internment Camp,"[M. Pietruschka, Franz Stampfl, W.A.B., H. W. ...",NaN,NaN,NaN,Concert,1940-01-01,Other Camp
8,Sergeant Snow White,musical revue (three-act pantomime/wartime sat...,1943,"Union Theatre, University of Melbourne (Univer...","Melbourne, Victoria, Australia","[Doc K. Sternberg, A. P. Schmitz, Max Lewinsky...",NaN,NaN,NaN,Revue/Variety,1943-01-01,Melbourne
9,Journey's End,full-length play,Not specified,None,Not specified,"[M. Bittermann, Bernard Joseph, David Rosentha...",NaN,NaN,NaN,Theatrical,NaT,None


### 10.  Tidy Data:  Working with Lists in Cells

In our  data, `person` names live in *three* different places in a record, each harder to harvest than the last:

1. **`Complete credits`** — every record has this: a flat, already-cleaned list of everyone mentioned anywhere in that record (parenthetical uncertainty notes like `"(possibly ...)"` already stripped out). No role information, but reliable and always present.

- We could use this for some charts that reveal the number of times a given person took part in some event, or to craft a network of related events (according to who took part).  But the cells are `lists` of names!

2. **`overall_credits`** (and, on some records, `performers`, `performers_list`, or `cast`) — this is where the actual *hierarchy of roles* lives: a dictionary mapping a role label (e.g. `"stage_manager"`, `"cast"`, `"orchestra_members"`) to the person(s) in that role. 

- However contents of this column ries per role: a single name (string), several names (list), or even a nested sub-grouping (a dictionary), such as pianists split out by season. Harvesting this requires handling all three shapes.

3. Even more difficult, in the full data, individual names are mentioned **inside** fields such as `acts`, `songs`, etc.  

- These are free-text, per-scene cast blurbs like `"Prince Charming - Eric Liffmann; The Witch/Grandma - George Blank"`. This is the messiest layer (would need regex or NLP to parse reliably) and we won't fully parse it here — it's left as an extension exercise.


#### Tidy Data Principles

Hadley Wickham famously formulated principles of what he calls "Tidy Data".  Tidy Data, in brief:

You can read more in Hadley Wickham's paper on Tidy Data here:  https://www.jstatsoft.org/article/view/v059i10

- Each variable forms a column (we already found ways to 'unnest' the JSON to deal with this)

- Each observation forms a row (here we have more work to do!)

- Each type of observational unit forms a table (to the extent that our data are all 'performances', we are already in reasonable shape for this)

Beyond having a largely standardized format for datasets, making your data "tidy" will massively simplify your work in Pandas, particularly as we try to find, filter, and group data in various ways.  All of Pandas built-in tools to parse, analyze, and visualize your data will work best when your data is organized following these principles.

The "Complete credits" column is a good place to start tidying.   Let's say that we are interested to find the number of times each performer took part in any event.  Currently is not so easy to do:  each "Complete credits" cell is in fact a `list` of names!



In [269]:
# the complete personnel for the FIRST event in our table--this is just the first FIVE names of many!
concert_df_brief.iloc[0]['Complete credits'][0:5]

['Doc K. Sternberg',
 'Ray Martin',
 'Jonny Flynn',
 'Rudolf Laqueur',
 'Herbert Voss']

There is a Pandas/Python function that will `explode` that list of strings into a Series (which is the equivalent of a Pandas column)


In [270]:

exploded_names = concert_df_brief['Complete credits'].explode()
exploded_names


0                                    Doc K. Sternberg
0                                          Ray Martin
0                                         Jonny Flynn
0                                      Rudolf Laqueur
0                                        Herbert Voss
                           ...                       
24                         'Allanne Stuart'nFryitte''
24                            'Rexinne Watsonto Nyte'
24                                     'Paul Reveere'
24    'Baron Heindrich Isaacstein von Reichardscraad'
25                                                NaN
Name: Complete credits, Length: 309, dtype: object

This in turn we can use with `value_counts` to get count of each time a given person took part in an event!


In [271]:
exploded_names.value_counts()

Complete credits
S. Cohn                                            6
Emil Wittenberg                                    6
H. W. Katz                                         5
M. Pietruschka                                     4
Ray Martin                                         4
                                                  ..
Löwenhart                                          1
Wolff                                              1
Rosen                                              1
Adler                                              1
'Baron Heindrich Isaacstein von Reichardscraad'    1
Name: count, Length: 242, dtype: int64

We can in fact do the same thing with the entire dataframe transforming the `wide` format (in which many items appear inside a list in our chosen column) into what Wickham would call `long` format data.

Now *each performer* appears in a single row *for each event in which they took part*.  The DF will grow  longer now 300+ rows instead of 240+.  But with ONE performer per row in the 'Complete credits' column, we can now analyze and visualize our data in new ways!  



In [272]:
concerts_exploded = concert_df_brief.explode('Complete credits')
# an some extra code to strip out any white space left behind after the split:

concerts_exploded['Complete credits'] = concerts_exploded['Complete credits'].str.strip()
concerts_exploded

,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano,type_category,date_time,place_cleaned
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Doc K. Sternberg,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Ray Martin,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Jonny Flynn,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Rudolf Laqueur,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Herbert Voss,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
...,...,...,...,...,...,...,...,...,...,...,...,...
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Allanne Stuart'nFryitte'',NaN,NaN,NaN,Theatrical,1941-01-01,None
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Rexinne Watsonto Nyte',NaN,NaN,NaN,Theatrical,1941-01-01,None
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Paul Reveere',NaN,NaN,NaN,Theatrical,1941-01-01,None
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Baron Heindrich Isaacstein von Reichardscraad',NaN,NaN,NaN,Theatrical,1941-01-01,None


In [273]:
concerts_exploded.iloc[0]['Complete credits']

'Doc K. Sternberg'

Filtering and Finding based on some condition

For instance, all the events in which a particular person took part:

In [274]:
name = 'Doc K. Sternberg'

concerts_exploded[concerts_exploded['Complete credits'] == name]

,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano,type_category,date_time,place_cleaned
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Doc K. Sternberg,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
8,Sergeant Snow White,musical revue (three-act pantomime/wartime sat...,1943,"Union Theatre, University of Melbourne (Univer...","Melbourne, Victoria, Australia",Doc K. Sternberg,NaN,NaN,NaN,Revue/Variety,1943-01-01,Melbourne


In [275]:
# filter by date:
year = '1941'

concerts_exploded[concerts_exploded['date'] == year]

,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano,type_category,date_time,place_cleaned
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Doc K. Sternberg,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Ray Martin,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Jonny Flynn,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Rudolf Laqueur,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Herbert Voss,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay
...,...,...,...,...,...,...,...,...,...,...,...,...
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Allanne Stuart'nFryitte'',NaN,NaN,NaN,Theatrical,1941-01-01,None
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Rexinne Watsonto Nyte',NaN,NaN,NaN,Theatrical,1941-01-01,None
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Paul Reveere',NaN,NaN,NaN,Theatrical,1941-01-01,None
24,The Ceramic Reparteg Club Presents: A Fruity M...,comic amateur play (one-act parody melodrama),1941,None,Not specified,'Baron Heindrich Isaacstein von Reichardscraad',NaN,NaN,NaN,Theatrical,1941-01-01,None


In [276]:
# or both conditions:
concerts_exploded[(concerts_exploded['date'] == year) & (concerts_exploded['Complete credits'] == name)]


,title,type,date,venue,location,Complete credits,overall_credits_written_produced_directed_by,overall_credits_music,overall_credits_musical_arrangements_and_piano,type_category,date_time,place_cleaned
0,Snowhite Joins Up,musical revue (pantomime/topical satire),1941,Camp theatre (exact hall not specified),"Hay Internment Camp, New South Wales, Australia",Doc K. Sternberg,Doc K. Sternberg,Ray Martin,"[Jonny Flynn, Rudolf Laqueur, Herbert Voss]",Revue/Variety,1941-01-01,Hay


In [277]:
places_event_types = concert_df_brief.groupby(['type_category', 'place_cleaned']).size().reset_index(name='count')
places_event_types

,type_category,place_cleaned,count
0,Concert,Hay,2
1,Concert,Melbourne,2
2,Concert,Other Camp,2
3,Concert,Sandwich,1
4,Concert,Tatura,2
5,Other,Hay,2
6,Other,Melbourne,1
7,Other,Tatura,1
8,Revue/Variety,Dunera,2
9,Revue/Variety,Hay,1


## Visualizing the archive

With `concerts_df` in hand, a few quick charts with **Plotly Express** (`px`) show the distribution of events by type, place, and time. Plotly Express charts are interactive by default (hover for details, zoom, pan) — handy for exploring a small archive like this one live in a workshop.

In [278]:

year_counts = concert_df_brief["date"].value_counts().sort_index().reset_index(name="count")
fig = px.bar(
    year_counts,
    x="date", y="count",
    title="Events by year",
    labels={"date": "Year", "count": "Number of events"},
)
fig.update_xaxes(type="category")
fig.show()

In [279]:
fig = px.bar(places_event_types,
    x="type_category", 
    y="count",
    color='place_cleaned',
    title="Events by type and place",
    labels={"type_category": "Event type", "count": "Number of events"},
)
fig.show()


In [280]:
concert_df_brief['count'] = concert_df_brief.groupby(['date', 'place_cleaned'])['title'].transform('count').fillna(1)

fig = px.scatter(
    concert_df_brief,
    x="date", y="place_cleaned",
    color="type_category",
    size="count",
    hover_name="title",
    title="Events over time, by place and type",
)
fig.update_xaxes(type="category")
fig.show()


## Setting up for the network stage

The next step beyond this notebook is a **bi-nodal network**: two kinds of nodes (people and places), with an edge whenever a person was involved in an event at that place. That graph itself will be built with `networkx` and rendered with `pyvis`, using code shared separately — this notebook's job is just to produce a clean edge list ready to hand off.

We build it by joining `credits_long` (title → person, from `Complete credits`) against `concerts_df` (title → place), then dropping the title and de-duplicating — a person who appears in multiple songs within the same event should only produce one edge to that event's place.

In [281]:
weighted_pairs = concerts_exploded[['Complete credits', 'place_cleaned']].groupby(['Complete credits', 'place_cleaned']).size().reset_index(name="weight")

# concerts_exploded

## A Bi-Nodal Network of People and Places

A network is just **nodes** (entities) connected by **edges** (relationships), optionally with **weights** on those edges indicating how strong a relationship is. So far every network we might build from this archive has had one *kind* of node. Here we build one with **two kinds at once**: people and places — a "bi-nodal" (or *bipartite*) network. As the general network tutorial puts it: "You could even have different kinds of nodes in the same network... distinguished by color or shape." That's exactly what we do below: people are one color, places are another.

The edges themselves come straight from `person_place_edges`-style data we already built above (`credits_long` joined to `concerts_df`) — an edge connects a person to a place whenever that person is credited on an event that happened there. This time we'll also count *how many* events link each person to each place, and use that count as the edge **weight**.

### Building the graph with NetworkX

We add the two node types in separate calls to `G.add_nodes_from()`, each tagged with a `color` (what pyvis will actually draw), a `bipartite` flag (the NetworkX-standard way of marking a graph's two node sets — useful if you later want NetworkX's own bipartite algorithms), and a `node_type` label (handy for filtering later, e.g. `[n for n, d in G.nodes(data=True) if d["node_type"] == "place"]`).

Edges then come straight from `person_place_counts`, with the event count carried over as both the NetworkX `weight` attribute and pyvis's `width` attribute (thicker edge = more shared events). Node `size` is scaled by degree — the same *centrality* idea from the general tutorial: people or places connected to many others end up bigger and get pulled toward the center once physics is applied.

In [282]:
import networkx as nx

#Two colors for our two kinds of nodes
PERSON_COLOR = "#4C72B0"  # blue
PLACE_COLOR = "#DD8452"   # orange

G = nx.Graph()

people = weighted_pairs["Complete credits"].unique()
places = weighted_pairs["place_cleaned"].unique()

G.add_nodes_from(people, bipartite=0, node_type="person", color=PERSON_COLOR)
G.add_nodes_from(places, bipartite=1, node_type="place", color=PLACE_COLOR, shape="square")

for _, row in weighted_pairs.iterrows():
    G.add_edge(row["Complete credits"], row["place_cleaned"], weight=int(row["weight"]))

for _, _, edge_data in G.edges(data=True):
    edge_data["width"] = edge_data["weight"]

#Scale node size and add a hover label based on degree (centrality)
for node, node_data in G.nodes(data=True):
    degree = G.degree(node)
    node_data["size"] = 10 + 3 * degree
    node_data["title"] = f"{node} ({node_data['node_type']}, {degree} connection{'s' if degree != 1 else ''})"

print(f"{G.number_of_nodes()} nodes ({len(people)} people, {len(places)} places), "
      f"{G.number_of_edges()} edges. Bipartite: {nx.is_bipartite(G)}")


230 nodes (224 people, 6 places), 260 edges. Bipartite: True


### Rendering with Pyvis

With ~250 nodes, the default physics leaves everything clumped in the middle — illegible. Setting `solver: forceAtlas2Based` (the same option used in the general tutorial) spreads the graph out so clusters of people around a shared place become visible, and lets you drag nodes around interactively once rendered.

`cdn_resources="in_line"` bundles Pyvis's JS/CSS directly into the output HTML — slightly larger file, but it means the graph still renders with no internet connection and doesn't hit the Chrome/Safari display issues Pyvis warns about under the default `"local"` setting.

In [283]:
from pyvis import network as net

person_place_network = net.Network(
    notebook=True,
    width="900px",
    height="900px",
    bgcolor="#111111",
    font_color="white",
    cdn_resources="in_line",
)

person_place_network.set_options("""
{
  "physics": {
    "enabled": true,
    "forceAtlas2Based": {
      "springLength": 100
    },
    "solver": "forceAtlas2Based"
  }
}
""")

person_place_network.from_nx(G)
person_place_network.show("person_place_network.html")


person_place_network.html


### Optional: Louvain Community Detection

Since people here only connect to places (never to each other directly), Louvain communities will mostly just rediscover "people tied to the same place" — but it's worth running anyway, because it's precisely the people credited at *more than one* place (e.g. someone who moved from Hay to Tatura to Melbourne, following the pattern the archive overview describes) who end up as bridges between communities, visually pulling two place-clusters together. Those bridge figures are often the most historically interesting nodes in the whole graph.

This cell needs the `python-louvain` package (`pip install python-louvain`, imported as `community`) — it's optional, so the cell checks for it rather than failing the whole notebook if it isn't installed.

In [249]:
try:
    from community import community_louvain
    from copy import deepcopy

    def add_communities(graph):
        graph = deepcopy(graph)
        partition = community_louvain.best_partition(graph)
        nx.set_node_attributes(graph, partition, "group")
        return graph

    G_communities = add_communities(G)
    print(f"{len(set(nx.get_node_attributes(G_communities, 'group').values()))} communities detected.")

    community_network = net.Network(
        notebook=True, width="900px", height="900px",
        bgcolor="#111111", font_color="white", cdn_resources="in_line",
    )
    community_network.set_options("""
    {
      "physics": {
        "enabled": true,
        "forceAtlas2Based": {"springLength": 100},
        "solver": "forceAtlas2Based"
      }
    }
    """)
    community_network.from_nx(G_communities)
    community_network.show("person_place_network_louvain.html")

except ImportError:
    print("python-louvain isn't installed. Run `pip install python-louvain` and re-run this cell to see communities.")


6 communities detected.
person_place_network_louvain.html
